In [1]:
# Reviewer: Anan Yablonko.
import numpy as np
from matplotlib import pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def loss(y, z):
    return -(y * np.log(z) + (1 - y) * np.log(1 - z)).mean()


def regularized_loss(y, z, w, l1_reg):
    return loss(y, z) + l1_reg * np.sum(np.abs(w))


def predict(x, w, b):
    return np.matmul(x, w) + b


def loss_derivative_weight(x, y, w, l1_reg, z):
    gradient = ((z - y).reshape(-1, 1) * x).mean(axis=0)
    return gradient + l1_reg * np.sign(w)


def loss_derivative_intercept(y, z):
    return (z - y).mean(axis=0)


def update(w, gradient, lr):
    return w - (gradient * lr)

In [2]:
def gradient_decent(x, y, w, lr=0.01, reg=0.01, n_iter=1000, stop=10, tolerance=1e-8, verbose=False):
    b, i = 0, 0
    losses = []
    best_loss = np.inf
    count = 0
    for i in range(n_iter):
        yhat = predict(x, w, b)
        sig = sigmoid(yhat)
        log_loss = regularized_loss(y, sig, w, reg)
        if log_loss < best_loss - tolerance:
            best_loss = log_loss
            count = 0
        else:
            count += 1
        if count == stop:
            break
        losses.append(log_loss)
        grad_w = loss_derivative_weight(x, y, w, reg, sig)
        grad_b = loss_derivative_intercept(y, sig)
        w = update(w, grad_w, lr)
        b = update(b, grad_b, lr)
        if verbose and i % n_iter / 100 == 0:
            print(f"Iteration {i} - Loss: {log_loss}")
    if verbose:
        plt.plot(losses)
        print(f"stopped after {i} iterations")
    return w, b, losses[-1]

In [3]:


reg = 0.01
n_samples = 100000
n_features = 12
binary = 2
cluster_std = 25.4
random_state = 1989
runs = 200000
learning_rate = 0.01

data = make_blobs(n_samples=n_samples, n_features=n_features, centers=binary, cluster_std=cluster_std,
                  random_state=random_state)
x, y = data[0], data[1]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=random_state)
weights = np.zeros(x.shape[1])
best_w, best_b, best_loss = gradient_decent(x_train, y_train, weights, learning_rate, reg, runs)

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

y_predict = predict(x_test, best_w, best_b) >= 0.5
print(classification_report(y_predict, y_test))
confusion_matrix(y_predict, y_test)

              precision    recall  f1-score   support

       False       0.84      0.62      0.72     20446
        True       0.48      0.76      0.59      9554

    accuracy                           0.66     30000
   macro avg       0.66      0.69      0.65     30000
weighted avg       0.73      0.66      0.68     30000



array([[12706,  7740],
       [ 2339,  7215]])